# Emotion Classification - Experiment 1 (Sharvan)

**Tasks 3 & 4 - model loading, Kaggle GPU training, W&B tracking**

My contribution to the group pipeline focuses on getting the evaluation right. The first set of
team notebooks only logged accuracy, but the rubric for Task 4 asks for training loss, validation
loss, accuracy **and F1**. So I rebuilt the training loop around the assignment's `compute_metrics`
template (sklearn `accuracy_score` + weighted `f1_score`) and ran two versions that differ by a
single hyperparameter - the per-device batch size - so the W&B comparison is clean.

This is **Experiment 1**: batch size = 32 (the larger-batch baseline).

In [ ]:
!pip install -q transformers datasets scikit-learn wandb

## 1. Load secrets from Kaggle

Tokens are never hardcoded. I pull `WANDB_API_KEY` and `HF_TOKEN` from Kaggle Secrets
(Add-ons -> Secrets) and log in to both services. The W&B project name matches the rest of
the team so all of our runs land on the same dashboard for the Task 8 comparison.

In [ ]:
import os
import wandb
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
login(token=secrets.get_secret("HF_TOKEN"))
wandb.login()

# Same project as the rest of Group 14 so every run is comparable on one dashboard
os.environ["WANDB_PROJECT"] = "mlops-emotion-classification"

## 2. Load the data and the model

Dataset is `dair-ai/emotion` (6 classes, ~16k train rows). I read the number of labels straight
from our committed `data/id2label.json` so the classifier head size can never drift out of sync
with the mapping. Model is `distilbert-base-uncased` - small enough (~260MB) to fine-tune inside
the free T4 quota. I take a seeded 8k/2k subset to keep the run short.

Note: upload `data/id2label.json` as a Kaggle dataset (named `id2label`) and attach it, or point
the path below at wherever you mount it.

In [ ]:
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# id2label drives num_labels - single source of truth for the label space
with open("/kaggle/input/id2label/id2label.json") as f:
    id2label = {int(k): v for k, v in json.load(f).items()}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)
print(f"Loaded {num_labels} labels: {list(id2label.values())}")

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

dataset = load_dataset("dair-ai/emotion")
print(dataset)


def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)


tokenized = dataset.map(tokenize, batched=True)
train_ds = tokenized["train"].shuffle(seed=42).select(range(8000))
eval_ds = tokenized["validation"].shuffle(seed=42).select(range(2000))
print(f"train={len(train_ds)}  eval={len(eval_ds)}")

## 3. Train and track on W&B

The config dict logged to W&B records every hyperparameter for the run. `compute_metrics`
returns accuracy and weighted F1; the Trainer already streams training loss and `eval_loss`,
so all four required metrics show up on the dashboard. `report_to="wandb"` wires it together.

**Experiment 1 config:** lr 2e-5, **batch size 32**, 3 epochs, weight decay 0.01.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score
from transformers import TrainingArguments, Trainer

wandb.init(
    project="mlops-emotion-classification",
    name="sharvan-exp1-bs32",
    config={
        "model": model_name,
        "epochs": 3,
        "batch_size": 32,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "max_length": 128,
        "train_samples": len(train_ds),
        "version": "v1",
        "platform": "Kaggle",
    },
)


def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
    }


training_args = TrainingArguments(
    output_dir="./results_exp1",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=25,
    report_to="wandb",
    run_name="sharvan-exp1-bs32",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())
wandb.finish()